# Notebook 2: Drug 3D Feature Extraction (GeoGNN via PaddleHelix)

**Purpose:** Extract 3D spatial drug features using GeoGNN, which encodes atom-bond and
bond-angle graphs derived from 3D molecular conformations generated by RDKit MMFF94.

**Input:** `data/clf/unique_drugs.csv` — unique drug SMILES strings from the DrugBank dataset.

**Output:** `bisai_HY_QS_3D_emb_64.npy` — a dictionary mapping each drug SMILES to a
64-dimensional 3D feature vector.

**Method:** SMILES strings are first filtered to remove molecules that cause deadlocks
during 3D embedding (oversized, heavily fused, or known problematic structures).
Filtered molecules are then processed by the PaddleHelix GeoGNN model.

> **Note:** This notebook requires PaddlePaddle and PaddleHelix, installed into a
> Miniconda Python 3.8 environment to satisfy compatibility requirements.

## Step 0: Mount Google Drive and Restore Project Files

In [ ]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

In [ ]:
import shutil
# Update zip_path to match the actual backup filename in your Google Drive
zip_path = "/content/drive/MyDrive/EviDTI_Protein_and_Drug_2D_Feature_Backup_20260301_0827.zip"
extract_path = "/content/EviDTI-main"

print("Extracting project files...")
shutil.unpack_archive(zip_path, extract_path)
print("Extraction complete.")

## Step 1: Install PaddlePaddle and PaddleHelix (Python 3.8 environment)

PaddleHelix requires Python 3.8 and a specific version of PaddlePaddle.
We install Miniconda to create a compatible environment, then install
PaddleHelix from source (the pip version is incomplete and missing key modules).

In [ ]:
!wget -qO miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh
!bash ./miniconda.sh -b -f -p /opt/conda

In [ ]:
print("Installing dependencies into Miniconda Python 3.8 environment...")

# Install scientific dependencies first to prevent paddlehelix from overwriting them
!env PYTHONPATH= /opt/conda/bin/pip install scikit-learn networkx pandas rdkit -i https://pypi.tuna.tsinghua.edu.cn/simple

# Install PaddlePaddle GPU
!env PYTHONPATH= /opt/conda/bin/pip install paddlepaddle-gpu==2.5.2 -i https://pypi.tuna.tsinghua.edu.cn/simple

# Install PaddleHelix with --no-deps to prevent dependency conflicts
!env PYTHONPATH= /opt/conda/bin/pip install paddlehelix --no-deps -i https://pypi.tuna.tsinghua.edu.cn/simple

print("Dependencies installed.")

In [ ]:
print("Installing PGL (Paddle Graph Learning)...")
!env PYTHONPATH= /opt/conda/bin/pip install pgl -i https://pypi.tuna.tsinghua.edu.cn/simple
print("PGL installed.")

In [ ]:
print("Step 1: Uninstalling pip version of paddlehelix (incomplete)...")


## Step 2: Filter SMILES

Before 3D embedding, molecules are filtered using thresholds refined through iterative testing.
Molecules that are too large, too heavily fused, or match known problematic substructures
are replaced with methane (`C`) as a zero-vector placeholder.

**Filtering criteria:**
- Blacklisted substrings (known deadlock-causing structures)
- SMILES string length > 110
- Macrocyclic peptides containing disulfide bonds (`CSSC`)
- Heavy atom count > 100
- Ring count >= 5
- Invalid SMILES (unparseable by RDKit)

In [ ]:
import pandas as pd
import os
import shutil
from rdkit import Chem

# Step 1: Clean up any previous working directories
folders_to_clean = ["/content/EviDTI-main/smiles_chunks", "/content/EviDTI-main/smiles_input", "/content/EviDTI-main/smiles_final"]
for folder in folders_to_clean:
    if os.path.exists(folder):
        shutil.rmtree(folder)

new_folder = "/content/EviDTI-main/smiles_final"
os.makedirs(new_folder)

# Step 2: Load SMILES from unique_drugs.csv
csv_path = "/content/EviDTI-main/data/clf/unique_drugs.csv"
df = pd.read_csv(csv_path)
smiles_list = df['smile'].dropna().tolist()

# Filtering thresholds refined through iterative testing:
MAX_ATOMS = 100  # Max heavy atom count
MAX_RINGS = 5    # Max number of rings
MAX_LEN = 110

# Blacklist: SMILES substrings known to cause deadlocks during 3D embedding
blacklist = [
    "C1C[C@H](N(C1)C(=O)[C@@H]2CSSC",  # large cyclic peptide with disulfide bond
    "CCSCC1=CC2=C(C=C1)N3[C@@H]4C",     # fused polycyclic compound
    "CC(=O)NC(=NCCC[C@H]1C(=O)N2CCC[C@@H]2C(=O)N[C@H]3",  # long cyclic peptide
    "CCCNC=N(N)NC(=O)NC"
]

processed_smiles = []
stats = {'blacklist': 0, 'large': 0, 'fused': 0, 'macro': 0, 'len': 0, 'invalid': 0, 'passed': 0}

print(f"Filtering SMILES (max length: {MAX_LEN})...")

for s in smiles_list:
    s_str = str(s).strip()
    is_bad = False

    # Filter 1: Blacklist substring match
    if any(b in s_str for b in blacklist):
        is_bad = True
        stats['blacklist'] += 1

    # Filter 2: String length and macrocyclic peptide pattern
    if not is_bad:
        if len(s_str) > MAX_LEN:
            is_bad = True
            stats['len'] += 1
        elif "CSSC" in s_str:  # Macrocyclic peptide with disulfide bond
            is_bad = True
            stats['macro'] += 1

    # Filter 3: RDKit-based structural properties
    if not is_bad:
        mol = Chem.MolFromSmiles(s_str)
        if mol:
            num_atoms = mol.GetNumHeavyAtoms()
            num_rings = mol.GetRingInfo().NumRings()

            if num_atoms > MAX_ATOMS:
                is_bad = True
                stats['large'] += 1
            elif num_rings >= MAX_RINGS:
                is_bad = True
                stats['fused'] += 1
        else:
            is_bad = True
            stats['invalid'] += 1

    if is_bad:
        processed_smiles.append("C")  # Replace with methane (zero-vector placeholder)
    else:
        processed_smiles.append(s_str)
        stats['passed'] += 1

# Step 3: Save all filtered SMILES to a single output file
output_path = os.path.join(new_folder, "all_drugs_cleaned.txt")
with open(output_path, "w") as f:
    for s in processed_smiles:
        f.write(s + "\n")

print("\n--- Filtering Report ---")
print(f"Passed:          {stats['passed']}")
print(f"Blacklisted:     {stats['blacklist']}")
print(f"Too long:        {stats['len']}")
print(f"Macrocyclic:     {stats['macro']}")
print(f"Too many atoms:  {stats['large']}")
print(f"Too many rings:  {stats['fused']}")
print(f"Invalid SMILES:  {stats['invalid']}")

print(f"Filtered SMILES saved to: {output_path}")

## Step 3: Run 3D Feature Extraction

In [ ]:
%cd /content/EviDTI-main
!env PYTHONPATH= /opt/conda/bin/python extract_drug_3d_emb.py --data_path ./smiles_final --batch_size 256

## Step 4: Save Results to Google Drive

In [ ]:
from google.colab import drive, files
import shutil
import datetime
import os

print("Backing up project to Google Drive...")


# Key output file path
source_dir = "/content/EviDTI-main"

drug_3d_feat = f"{source_dir}/bisai_HY_QS_3D_emb_64.npy"



# Verify output file exists
if os.path.exists(drug_3d_feat):
    d_size_mb = os.path.getsize(drug_3d_feat) / (1024**2)
    print(f"Drug 3D features confirmed: {d_size_mb:.2f} MB")

    # Pack and upload entire project to Google Drive
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    backup_name = f"EviDTI_Full_Project_Backup_{timestamp}"
    destination_zip = f"/content/drive/MyDrive/{backup_name}"

    
    print("Packing project...")

    shutil.make_archive(destination_zip, 'zip', source_dir)

    print(f"Backup saved: {backup_name}.zip")
else:
    print("Warning: drug 3D feature file not found, please check the path.")